In [50]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os


In [51]:
#!git clone https://github.com/recsyspolimi/RecSys_Course_AT_PoliMi

import os
!pwd
current_directory = os.getcwd()
if current_directory.endswith("RecSys_Course_AT_PoliMi"):
    print("Already in the correct directory.")
    pass
else:
    os.chdir("/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/RecSys_Course_AT_PoliMi")

!pwd
!python run_compile_all_cython.py

/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/Results
/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/RecSys_Course_AT_PoliMi
zsh:1: command not found: python


In [52]:
"""import os
import time 
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.sparse as sps
import matplotlib.pyplot as pyplot
%matplotlib inline

from sklearn.model_selection import KFold
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from skopt.space import Real, Integer, Categorical
from Evaluation.Evaluator import EvaluatorHoldout
from HyperparameterTuning.SearchBayesianSkopt import SearchBayesianSkopt"""

# from Recommenders.SLIM.SLIM_BPR_Python import SLIM_BPR_Python

import os

# --- STEP 1: HOTFIX FOR RECSYS LIBRARY ---
# This block programmatically fixes the "SearchAbstractClass.py" file
# by removing the reference to the non-existent _ArrayMemoryError.
file_path = "HyperparameterTuning/SearchAbstractClass.py"

if os.path.exists(file_path):
    with open(file_path, "r") as f:
        lines = f.readlines()
    
    with open(file_path, "w") as f:
        for line in lines:
            # Comment out the import of _ArrayMemoryError (whether from core or exceptions)
            if "import _ArrayMemoryError" in line:
                f.write("# " + line) 
            # Update the exception tuple to only use the standard MemoryError
            elif "MEMORY_ERROR_EXCEPTION_TUPLE =" in line:
                f.write("MEMORY_ERROR_EXCEPTION_TUPLE = (MemoryError, )\n")
            else:
                f.write(line)
    print(f"Successfully patched {file_path}")
else:
    print(f"Could not find {file_path}. Make sure your working directory is correct.")
# ------------------------------------------


# --- STEP 2: YOUR IMPORTS ---
import time 
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.sparse as sps
import matplotlib.pyplot as pyplot
%matplotlib inline

from sklearn.model_selection import KFold
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from skopt.space import Real, Integer, Categorical
from Evaluation.Evaluator import EvaluatorHoldout

# Note: We removed the line importing _ArrayMemoryError here because it no longer exists.

from HyperparameterTuning.SearchBayesianSkopt import SearchBayesianSkopt

print("Imports successful!")

Successfully patched HyperparameterTuning/SearchAbstractClass.py
Imports successful!


In [53]:
df_train = pd.read_csv("data_train.csv")
df_test_user = pd.read_csv("data_target_users_test.csv")

In [54]:
"""
Created on 28 June 2017
Updated on 28 November 2020

@author: Maurizio Ferrari Dacrema
"""

# COPIED FROM GITHUB BECAUSE OF COMPATIBILITY ERROR ON NP.FLOAT


'\nCreated on 28 June 2017\nUpdated on 28 November 2020\n\n@author: Maurizio Ferrari Dacrema\n'

In [55]:
#df_train = df_train.iloc[:-1]

In [56]:
def split_train_in_five_percentage_global_sample(URM_all, train_percentages):
    """
    The function splits an URM in five matrices based on provided percentages.
    :param URM_all: The full URM matrix
    :param train_percentages: A list of percentages (must sum to 1.0)
    :return: A list of 5 sparse matrices
    """

    import numpy as np
    from scipy.sparse import coo_matrix
    from Data_manager.IncrementalSparseMatrix import IncrementalSparseMatrix

    assert len(train_percentages) == 5, "You must provide exactly 5 percentages."
    assert abs(sum(train_percentages) - 1.0) < 1e-6, "Percentages must sum to 1.0."

    num_users, num_items = URM_all.shape

    # Builders for each of the 5 matrices
    builders = [
        IncrementalSparseMatrix(n_rows=num_users, n_cols=num_items, auto_create_col_mapper=False, auto_create_row_mapper=False)
        for _ in range(5)]

    URM_all_coo = coo_matrix(URM_all)

    # Shuffle indices
    indices_for_sampling = np.arange(URM_all.nnz, dtype=np.int32)
    np.random.shuffle(indices_for_sampling)

    # Calculate the number of interactions for each split
    split_sizes = [int(URM_all.nnz * percentage) for percentage in train_percentages]
    cumulative_sizes = np.cumsum(split_sizes)

    # Divide the indices into 5 groups
    indices_splits = [
        indices_for_sampling[cumulative_sizes[i - 1]:cumulative_sizes[i]] if i > 0 else indices_for_sampling[:cumulative_sizes[i]]
        for i in range(5)
    ]

    # Populate the builders
    for i, builder in enumerate(builders):
        builder.add_data_lists(
            URM_all_coo.row[indices_splits[i]],
            URM_all_coo.col[indices_splits[i]],
            URM_all_coo.data[indices_splits[i]],
        )

    # Convert to sparse matrices
    sparse_matrices = [builder.get_SparseMatrix() for builder in builders]

    # Ensure all outputs are in csr_matrix format
    sparse_matrices = [sp.csr_matrix(matrix) for matrix in sparse_matrices]

    return sparse_matrices

In [57]:
from scipy.sparse import coo_matrix

#valore 1 per ogni coppia (row, col)
data = [1] * len(df_train)
df_train["row"] = df_train["row"].astype(int)
df_train["col"] = df_train["col"].astype(int)

# matrice COO
URM_all = sp.csr_matrix((data, (df_train["row"], df_train["col"])))

In [58]:
train_percentages = [0.2, 0.2, 0.2, 0.2, 0.2]  # Cinque parti uguali

URM_parts = split_train_in_five_percentage_global_sample(URM_all, train_percentages)
URM_parts

[<Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>]

In [ ]:
## import time 

class SaveResults(object):
    
    def __init__(self):
        self.results_df = pd.DataFrame(columns=["result", "train_time (min)"])
    
    def __call__(self, optuna_study, optuna_trial):
        hyperparam_dict = optuna_trial.params.copy()
        hyperparam_dict["result"] = optuna_trial.values[0]
        
        # Retrieve the optimal number of epochs and training time from the "user attributes" of the trial
        #hyperparam_dict["epochs"] = optuna_trial.user_attrs["epochs"]
        hyperparam_dict["train_time (min)"] = optuna_trial.user_attrs["train_time (min)"]
        
        self.results_df.loc[len(self.results_df)] = hyperparam_dict
        
        
def objective_function_funksvd(optuna_trial):

                          
    start_time = time.time()
    scores = []
    for i in range(5):
        URM_combined = sum(URM_parts[j] for j in range(len(URM_parts)) if j != i)
        
        
        #Cambiare il modello qui sotto, insieme al range e ai parametri         
        recommender_instance = SLIM_BPR_Python(URM_combined)
        
        recommender_instance.fit(
                        topK = optuna_trial.suggest_int("topK", 1, 50),
                         learning_rate = optuna_trial.suggest_float("learning_rate", 1e-2, 9e-2, log=True),
                         lambda_i = optuna_trial.suggest_float("lambda_i", 1e-5, 5e-4, log=True),
                         lambda_j = optuna_trial.suggest_float("lambda_j", 1e-5, 2e-4, log=True),                         
                         # epochs = optuna_trial.suggest_int("epochs", 10, 50) 
                         )

        
        evaluator_test = EvaluatorHoldout(URM_parts[i], cutoff_list=[20])
        result, _ = evaluator_test.evaluateRecommender(recommender_instance)
        #print("prova = ", result["MAP"].values[0])
        #print(result)
        scores.append(result["RECALL"].values[0])
        #if result["MAP"].values[0] < 0.051:
        #    break
        
    # Add the number of epochs selected by earlystopping as a "user attribute" of the optuna trial
    # Qui ripristinate le epochs
    # epochs = recommender_instance.get_early_stopping_final_epochs_dict()["epochs"]
    # optuna_trial.set_user_attr("epochs", epochs) 

    
    optuna_trial.set_user_attr("train_time (min)", (time.time() - start_time)/60) 
    print(scores)
    return sum(scores) / len(scores)

In [60]:
import optuna
optuna_study = optuna.create_study(direction="maximize")
        
save_results = SaveResults()
        
optuna_study.optimize(objective_function_funksvd,
                      callbacks=[save_results],
                      n_trials = 25)

[I 2025-11-30 19:01:55,309] A new study created in memory with name: no-name-79c4340d-89e8-44ab-bca8-2c6ec45322c8


Epoch 1, Iteration 27095 in 0.81 seconds. Samples per second 33593.96
Epoch 2, Iteration 27095 in 0.71 seconds. Samples per second 38413.95
Epoch 3, Iteration 27095 in 0.70 seconds. Samples per second 38680.30
Epoch 4, Iteration 27095 in 0.70 seconds. Samples per second 38493.34
Epoch 5, Iteration 27095 in 0.71 seconds. Samples per second 38429.86
Epoch 6, Iteration 27095 in 0.71 seconds. Samples per second 38399.89
Epoch 7, Iteration 27095 in 0.71 seconds. Samples per second 38333.61
Epoch 8, Iteration 27095 in 0.71 seconds. Samples per second 38037.19
Epoch 9, Iteration 27095 in 0.71 seconds. Samples per second 38400.11
Epoch 10, Iteration 27095 in 0.71 seconds. Samples per second 38423.14
Epoch 11, Iteration 27095 in 0.71 seconds. Samples per second 38374.88
Epoch 12, Iteration 27095 in 0.71 seconds. Samples per second 38178.33
Epoch 13, Iteration 27095 in 0.71 seconds. Samples per second 38364.71
Epoch 14, Iteration 27095 in 0.71 seconds. Samples per second 38429.80
Epoch 15, Itera

[I 2025-11-30 19:03:58,151] Trial 0 finished with value: 0.22393301508902613 and parameters: {'topK': 32, 'learning_rate': 0.07476807390095361, 'lambda_i': 0.0001257438451384195, 'lambda_j': 2.495174920510822e-05}. Best is trial 0 with value: 0.22393301508902613.


[0.2231401828237922, 0.22392767359945093, 0.22251005907766355, 0.22554495487167256, 0.2245422050725514]
Epoch 1, Iteration 27095 in 0.71 seconds. Samples per second 38280.64
Epoch 2, Iteration 27095 in 0.71 seconds. Samples per second 38272.96
Epoch 3, Iteration 27095 in 0.71 seconds. Samples per second 38365.52
Epoch 4, Iteration 27095 in 0.71 seconds. Samples per second 38356.29
Epoch 5, Iteration 27095 in 0.71 seconds. Samples per second 38307.26
Epoch 6, Iteration 27095 in 0.71 seconds. Samples per second 38400.15
Epoch 7, Iteration 27095 in 0.71 seconds. Samples per second 38377.21
Epoch 8, Iteration 27095 in 0.71 seconds. Samples per second 38388.62
Epoch 9, Iteration 27095 in 0.71 seconds. Samples per second 38323.52
Epoch 10, Iteration 27095 in 0.70 seconds. Samples per second 38501.38
Epoch 11, Iteration 27095 in 0.71 seconds. Samples per second 38348.91
Epoch 12, Iteration 27095 in 0.71 seconds. Samples per second 38227.12
Epoch 13, Iteration 27095 in 0.70 seconds. Samples pe

[I 2025-11-30 19:05:56,195] Trial 1 finished with value: 0.21773627201246906 and parameters: {'topK': 7, 'learning_rate': 0.010034760615765964, 'lambda_i': 0.00021340411369952616, 'lambda_j': 1.1264790197017702e-05}. Best is trial 0 with value: 0.22393301508902613.


...

[0.2318024381627898, 0.23205635924644682, 0.23188185353442964, 0.2343668653243846, 0.23295330674291295]
Epoch 1, Iteration 27095 in 1.22 seconds. Samples per second 22219.73
Epoch 2, Iteration 27095 in 1.16 seconds. Samples per second 23404.07
Epoch 3, Iteration 27095 in 1.17 seconds. Samples per second 23090.78
Epoch 4, Iteration 27095 in 1.18 seconds. Samples per second 22890.79
Epoch 5, Iteration 27095 in 1.18 seconds. Samples per second 23041.14
Epoch 6, Iteration 27095 in 1.18 seconds. Samples per second 22943.55
Epoch 7, Iteration 27095 in 1.18 seconds. Samples per second 23020.00
Epoch 8, Iteration 27095 in 1.18 seconds. Samples per second 22964.18
Epoch 9, Iteration 27095 in 1.18 seconds. Samples per second 23004.85
Epoch 10, Iteration 27095 in 1.19 seconds. Samples per second 22820.26
Epoch 11, Iteration 27095 in 1.18 seconds. Samples per second 22869.30
Epoch 12, Iteration 27095 in 1.18 seconds. Samples per second 22979.96
Epoch 13, Iteration 27095 in 1.18 seconds. Samples pe

[I 2025-11-30 19:52:16,296] Trial 23 finished with value: 0.22707318990353578 and parameters: {'topK': 17, 'learning_rate': 0.031491006414930335, 'lambda_i': 2.4753508139068188e-05, 'lambda_j': 2.0342130173359135e-05}. Best is trial 22 with value: 0.23261216460219275.


[0.22731808792130076, 0.2261811166930286, 0.22655051896216277, 0.2278489987898711, 0.22746722715131557]
Epoch 1, Iteration 27095 in 1.18 seconds. Samples per second 22941.43
Epoch 2, Iteration 27095 in 1.17 seconds. Samples per second 23146.68
Epoch 3, Iteration 27095 in 1.17 seconds. Samples per second 23198.57
Epoch 4, Iteration 27095 in 1.17 seconds. Samples per second 23208.37
Epoch 5, Iteration 27095 in 1.17 seconds. Samples per second 23147.81
Epoch 6, Iteration 27095 in 1.17 seconds. Samples per second 23244.38
Epoch 7, Iteration 27095 in 1.17 seconds. Samples per second 23223.86
Epoch 8, Iteration 27095 in 1.16 seconds. Samples per second 23294.66
Epoch 9, Iteration 27095 in 1.17 seconds. Samples per second 23197.28
Epoch 10, Iteration 27095 in 1.17 seconds. Samples per second 23177.27
Epoch 11, Iteration 27095 in 1.17 seconds. Samples per second 23219.70
Epoch 12, Iteration 27095 in 1.16 seconds. Samples per second 23263.18
Epoch 13, Iteration 27095 in 1.17 seconds. Samples pe

[I 2025-11-30 19:55:27,363] Trial 24 finished with value: 0.23133185407023826 and parameters: {'topK': 10, 'learning_rate': 0.045526232818260716, 'lambda_i': 1.526133749597818e-05, 'lambda_j': 1.7264052297843806e-05}. Best is trial 22 with value: 0.23261216460219275.


[0.2321011763720834, 0.2302400314134527, 0.230455218710932, 0.23108826145951847, 0.2327745823952047]


In [61]:
optuna_study.best_trial.params

save_results.results_df

# 1. Crea il DataFrame completo dallo studio Optuna
df_trials = optuna_study.trials_dataframe()

print(df_trials.columns)

Index(['number', 'value', 'datetime_start', 'datetime_complete', 'duration',
       'params_lambda_i', 'params_lambda_j', 'params_learning_rate',
       'params_topK', 'user_attrs_train_time (min)', 'state'],
      dtype='object')


In [62]:
"""save_results.results_df

# 1. Crea il DataFrame completo dallo studio Optuna
df_trials = optuna_study.trials_dataframe()"""

"""
- topK: 
- learning_rate:
- lambda_i: 
- lambda_j: """

current_directory = os.getcwd()
if current_directory.endswith("Results"):
    print("Already in the correct directory.")
    pass
else:
    os.chdir("/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/Results")
!pwd

# 2. Seleziona le colonne specifiche
df_final = df_trials[['number', 'value', 'params_topK', 'params_learning_rate', 'params_lambda_i', 'params_lambda_j']].copy()

# 3. Rinomina le colonne per averle pulite nel CSV
df_final.columns = ['trial', 'recall', 'topK', 'learning_rate', 'lambda_i', 'lambda_j']

# 4. Salva su file CSV (senza l'indice di pandas)
df_final.to_csv("slimbpr_par_opt_results_v2.csv", index=False)

df_final = df_final.sort_values(by='recall', ascending=False)

print("Salvataggio completato. Ecco le prime righe:")
print(df_final.head())

/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/Results
Salvataggio completato. Ecco le prime righe:
    trial    recall  topK  learning_rate  lambda_i  lambda_j
22     22  0.232612     8       0.041330  0.000025  0.000019
12     12  0.232138    11       0.052141  0.000491  0.000167
17     17  0.232121     7       0.036390  0.000038  0.000015
7       7  0.232056     7       0.038506  0.000435  0.000013
21     21  0.231964     9       0.039468  0.000375  0.000015


In [63]:
best_index = save_results.results_df["result"].idxmax()
best_hyperparams = save_results.results_df.loc[best_index].to_dict()

del best_hyperparams["result"]
del best_hyperparams["train_time (min)"]
best_hyperparams

{}